# RegLens · Validation — AUROC on the matched MPRA benchmark

Runs the **credibility linchpin**: does the pretrained ChromBPNet model's variant score
(`|Δ log-counts|`) rank **functional** regulatory variants above **non-functional** ones
— on the *hard, matched* comparison — and does it beat CADD?

### Honest expectations (read first)
- Negatives are **matched within-element** (Kircher satMutMPRA): the model must separate
  functional from non-functional variants *in the same active regions*. This is HARD.
- **Do not expect 0.9.** A modest AUROC (~0.60–0.70) that **beats CADD** on this fair
  benchmark is a legitimate, strong, honest result. Report it straight either way —
  per-element and overall, with the class balance.
- This validates the **ENGINE** (the variant score). The **AGENT** (multi-agent
  interpretation) is a *separate* claim, validated by recovering rs1427407 / rs2814778
  mechanisms + the red-team catching artifacts. Don't conflate them.

### Two passes (don't let 33k variants become a Sunday-night surprise)
1. **Subset (BCL11A, ~1.5k variants)** → a real number fast + confirm the pipeline scales.
2. **Full 33k** on the GPU → per-element + overall AUROC.

**Requires a GPU runtime.**

In [ ]:
# --- GPU + install -----------------------------------------------------------
!nvidia-smi -L || echo 'NO GPU — switch Runtime to GPU.'
!pip -q install 'tensorflow>=2.12' pyfaidx typer numpy matplotlib
# Upload the RegLens repo zip (reglens_for_colab.zip) via the Files panel, then:
!unzip -oq /content/reglens_for_colab.zip -d /content/RegLens && pip -q install -e /content/RegLens
import os; os.chdir('/content/RegLens'); print('cwd', os.getcwd())

In [ ]:
# --- Genome + pretrained model ----------------------------------------------
import glob, os
os.makedirs('/content/ref', exist_ok=True)
%cd /content/ref
# hg38 via Google Cloud (Broad ref): resolves on Colab even when UCSC DNS fails,
# chr-named (chr1, chr2, ...) matching the benchmark, and ships its own .fai.
!wget -c -O hg38.fa     https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta
!wget -c -O hg38.fa.fai https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta.fai
!ls -lh hg38.fa
!python -c "import pyfaidx; g=pyfaidx.Fasta('hg38.fa'); print('faidx OK; chr2 len', len(g['chr2']))"
# ENCODE K562 ATAC ChromBPNet models (needed only for Pass 1/2, not the crossover).
![ -f ENCFF984RAF.tar.gz ] || wget -q -c -O ENCFF984RAF.tar.gz https://www.encodeproject.org/files/ENCFF984RAF/@@download/ENCFF984RAF.tar.gz
!mkdir -p encode_models && tar -xzf ENCFF984RAF.tar.gz -C encode_models
print(len(glob.glob('encode_models/**/*chrombpnet_nobias*.h5', recursive=True)), 'K562 folds')
%cd /content/RegLens

In [ ]:
# --- PASS 1: subset (BCL11A) — fast real number -----------------------------
from reglens.validation import load_labeled_variants, evaluate
from reglens.tools.chrombpnet_score import ChromBPNetScorer, load_backend
from reglens.report.render import render_validation
import time

bench = load_labeled_variants('data/benchmarks/kircher_mpra_grch38.cadd.tsv')
bcl11a = [v for v in bench if v.source == 'BCL11A']
print(f'BCL11A subset: {len(bcl11a)} variants, {sum(v.label for v in bcl11a)} positive')

# Fold + reverse-complement averaged backend over all folds.
scorer = ChromBPNetScorer(load_backend('/content/ref/encode_models', average_rc=True),
                          window_length=2114, model_name='K562-5fold+rc')
t0=time.time()
rep = evaluate(bcl11a, scorer, genome_path='/content/ref/hg38.fa', progress=True)
print(f'scored in {time.time()-t0:.0f}s')
print(render_validation(rep))

### CADD baseline (for the model-vs-CADD comparison)
The CADD web form truncates large VCFs. Instead, pull CADD PHRED **directly** from the
pre-scored whole-genome file over remote tabix — 100% coverage in seconds (needs `pysam`):
```python
!pip -q install pysam
from reglens.validation.cadd import annotate_from_remote_cadd
annotate_from_remote_cadd('data/benchmarks/kircher_mpra_grch38.tsv',
                          'data/benchmarks/kircher_mpra_grch38.cadd.tsv')
```
Then load `...cadd.tsv` in Pass 2 and the harness reports `baseline(CADD)` automatically.
(Already committed in the repo, so you can skip this and load the `.cadd.tsv` directly.)

In [ ]:
# --- PASS 2: full 33k (GPU) — per-element + overall --------------------------
# Heavy: 33k variants × (ref+alt) × fwd/RC × folds. Run on GPU; expect a while.
import time
t0=time.time()
full = evaluate_batched(bench, scorer, genome_path='/content/ref/hg38.fa', chunk_size=2000, progress=True)
print(f'scored {len(bench)} variants in {time.time()-t0:.0f}s')
print(render_validation(full))
import json; print(json.dumps(full.to_dict(), indent=2))

In [ ]:
# --- ROC money-shot figure (model vs CADD) ----------------------------------
from reglens.report.plot import plot_roc
plot_roc(full, '/content/roc_mpra.png')
from IPython.display import Image; Image('/content/roc_mpra.png')

In [ ]:
# --- Cell-type specificity + per-element figure -----------------------------
from reglens.validation.lineage import render_celltype_specificity
from reglens.report.plot import plot_per_element
print(render_celltype_specificity(full))
plot_per_element(full, '/content/per_element_auroc.png')
from IPython.display import Image; Image('/content/per_element_auroc.png')

## Reporting (honest)
- State **overall AUROC** and **per-element AUROC**, and the **class balance**
  (positive/negative counts) — `render_validation` prints all three.
- If model **beats CADD by a clear margin** on this matched benchmark → that's the headline.
- If it's **close**, report it straight. Matched within-element negatives is a hard task;
  a modest AUROC that edges CADD is a real, defensible result.
- Keep the engine claim (this AUROC) separate from the agent claim (mechanism recovery).

In [ ]:
# --- CROSSOVER experiment: HepG2 (hepatic) model on the SAME benchmark -------
# Double dissociation test: if K562 wins on hematopoietic elements and HepG2 wins on
# hepatic ones, the signal is cell-type-driven, not a model artifact. Self-contained:
# uses the K562 per-element AUROC from your completed run (no need to re-score K562).
import glob, time
from reglens.validation import load_labeled_variants, evaluate_batched
from reglens.tools.chrombpnet_score import ChromBPNetScorer, KerasChromBPNetBackend
from reglens.validation.lineage import crossover_summary, is_double_dissociation
from reglens.report.plot import plot_crossover

# K562 per-element AUROC from the full 33k run:
K562 = {'SORT1.2':0.584,'FOXE1':0.430,'SORT1-flip':0.638,'SORT1':0.586,'BCL11A':0.620,
'ZFAND3':0.637,'MYCrs6983267':0.634,'UC88':0.709,'MSMB':0.606,'TCF7L2':0.487,'RET':0.546,
'IRF6':0.533,'ZRSh-13':0.538,'MYCrs11986220':0.463,'PKLR-24h':0.794,'IRF4':0.548,
'ZRSh-13h2':0.564,'PKLR-48h':0.805,'GP1BA':0.729,'LDLR.2':0.679,'LDLR':0.691,'F9':0.642,
'HNF4A':0.611,'HBG1':0.663,'TERT-GSc':0.672,'TERT-GBM':0.672,'TERT-GAa':0.649,
'TERT-HEK':0.705,'HBB':0.684}

# HepG2 (hepatic) ATAC ChromBPNet model — ENCODE ENCFF137WCM (5 folds):
if not glob.glob('/content/ref/hepg2_models/**/*chrombpnet_nobias*.h5', recursive=True):
    !cd /content/ref && wget -q -c -O ENCFF137WCM.tar.gz https://www.encodeproject.org/files/ENCFF137WCM/@@download/ENCFF137WCM.tar.gz
    !cd /content/ref && mkdir -p hepg2_models && tar -xzf ENCFF137WCM.tar.gz -C hepg2_models
folds = sorted(glob.glob('/content/ref/hepg2_models/**/*chrombpnet_nobias*.h5', recursive=True))
print(len(folds), 'HepG2 folds')

bench = load_labeled_variants('/content/RegLens/data/benchmarks/kircher_mpra_grch38.tsv')
scorer = ChromBPNetScorer(KerasChromBPNetBackend(folds, average_rc=True),
                          window_length=2114, model_name='HepG2-5fold+rc')
t0 = time.time()
hep = evaluate_batched(bench, scorer, genome_path='/content/ref/hg38.fa', progress=True)
print(f'HepG2 scored in {time.time()-t0:.0f}s')

HEPG2 = {e: a for e, a, *_ in hep.per_source_auroc() if a is not None}
per = {'K562': K562, 'HepG2': HEPG2}
print('HepG2 per-element AUROC:', {e: round(a, 3) for e, a in HEPG2.items()})
s = crossover_summary(per)
print('crossover means:', s)
print('DOUBLE DISSOCIATION:', is_double_dissociation(s, 'K562', 'HepG2'))
plot_crossover(per, '/content/crossover.png')
from IPython.display import Image; Image('/content/crossover.png')

## The crossover (strongest result)
If the K562 model wins on hematopoietic elements **and** the HepG2 model wins on hepatic
elements, that's a **double dissociation** — swapping the cell-type model swaps which
elements it wins on. That is intervention-level evidence that the signal is genuinely
cell-type-driven. Report it honestly either way; a partial flip is still informative.